# Price Prediction Model Training
**Model:** Decision Tree Regressor
**Output:** `backend/app/ml_models/price_model.pkl`

**Dataset:** `data/raw/mandi_prices.csv` (already in repo)
For richer data: download from [AGMARKNET](https://agmarknet.gov.in/)

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv('../data/raw/mandi_prices.csv')
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
print(df.head())
print(f'Shape: {df.shape}')

In [ ]:
# Encode categorical columns
le_crop = LabelEncoder()
le_state = LabelEncoder()

df['crop_enc'] = le_crop.fit_transform(df['commodity'])
df['state_enc'] = le_state.fit_transform(df['state'])

# Save encoders for inference
import os
os.makedirs('../backend/app/ml_models', exist_ok=True)
joblib.dump(le_crop, '../backend/app/ml_models/le_crop.pkl')
joblib.dump(le_state, '../backend/app/ml_models/le_state.pkl')

FEATURES = ['crop_enc', 'state_enc', 'month']
TARGET = 'modal_price'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Train Gradient Boosting Regressor (better than plain Decision Tree)
model = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f'MAE:  ₹{mean_absolute_error(y_test, y_pred):.2f}')
print(f'R²:   {r2_score(y_test, y_pred):.4f}')

In [ ]:
# Visualise predictions vs actuals
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.scatter(y_test, y_pred, alpha=0.4, color='#5C7A52')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Price (₹/quintal)')
plt.ylabel('Predicted Price (₹/quintal)')
plt.title('Price Prediction: Actual vs Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Save model
joblib.dump(model, '../backend/app/ml_models/price_model.pkl')
print('price_model.pkl saved!')

# Quick test
loaded = joblib.load('../backend/app/ml_models/price_model.pkl')
sample = np.array([[0, 0, 4]])  # crop_id=0, state_id=0, month=April
print(f'Sample prediction: ₹{loaded.predict(sample)[0]:.0f}/quintal')